# STAT 404 Project: Search Algorithms and Maze Structure on Pathfinding Efficiency

**Group members:** Naufal Prasojo, Chriscenci Susanto, Firila Najma Wahidah, Reyhan Pamungkas  
**Lab section:** L1B

This notebook runs the split-plot analysis of the maze experiment. It reads `results.csv`, written by `404.py` (one row per maze and algorithm, 1,350 rows), so that file needs to be in the notebook's working directory.

## Experiment summary

| Component | Description |
|---|---|
| **Treatments** | Search algorithm (5 levels: BFS, Dijkstra, A\*, greedy best-first, random walk); maze size (20×20, 50×50, 100×100); loop density (5%, 15%, 30%) |
| **Experimental unit** | One maze instance (270 mazes), carrying size and loop density |
| **Blocking** | Generation method (recursive backtracking/DFS, randomized Prim's, randomized Kruskal's) |
| **Randomization** | Recorded random seeds per maze; randomized order of the 270 mazes; randomized order of the five algorithms on each maze |
| **Replication** | 10 independent mazes for each size × density × generation-method combination |

## Objectives

- Compare the efficiency of the pathfinding algorithms on grid mazes.
- Test whether maze structure (size, loop density, generation method) changes which algorithm performs best (algorithm × maze interactions).
- Quantify how much variation comes from the algorithm versus the maze itself.

## Data columns used

| Column | Meaning |
|---|---|
| `maze_id` | Unique ID for each of the 270 mazes |
| `gen_method` | `dfs`, `prim`, `kruskal` (block G) |
| `size` | 20, 50, 100 (S) |
| `loop_density` | 0.05, 0.15, 0.30 (D) |
| `algorithm` | `bfs`, `dijkstra`, `astar`, `greedy`, `random_walk` (A) |
| `frac_explored` | Response: proportion of the maze's cells explored (`nodes_explored / size^2`) |
| `success` | 1 if the goal was reached (0 = random walk hit its step cap) |

## Setup

In [ ]:
library(emmeans)   # install.packages("emmeans") if needed
library(MASS)      # for boxcox

options(repr.plot.width = 9, repr.plot.height = 5)   # plot size in Jupyter

## Load data

The response is `log(frac_explored)`.

In [ ]:
dat <- read.csv("results.csv")
dat$maze <- factor(dat$maze_id)
dat$S <- factor(dat$size)
dat$D <- factor(dat$loop_density)
dat$G <- factor(dat$gen_method)
dat$A <- factor(dat$algorithm,
                levels = c("bfs", "dijkstra", "astar", "greedy", "random_walk"))
dat$y <- log(dat$frac_explored)            # response: log(proportion of cells explored)
stopifnot(!any(is.na(dat$A)), nrow(dat) == 1350)

## 0. Data checks

Random-walk runs that hit the step cap are censored. Also shown: the mean proportion explored by algorithm and size.

In [ ]:
with(dat[dat$A == "random_walk", ], print(table(S, D, success)))
print(round(with(dat, tapply(frac_explored, list(A, S), mean)), 3))

## 1. Split-plot ANOVA with two error terms

- **Maze stratum:** G, S, D, S:D (whole-plot error, 259 df)
- **Within stratum:** A, A:S, A:D (subplot error, 1060 df)

In [ ]:
fit <- aov(y ~ G + S*D + A*(S + D) + Error(maze), data = dat)
summary(fit)

## 2. Interaction plots and boxplots

In [ ]:
interaction.plot(dat$S, dat$A, dat$y, xlab = "Size", ylab = "Mean log(proportion explored)", trace.label = "Algorithm")

In [ ]:
interaction.plot(dat$D, dat$A, dat$y, xlab = "Loop density", ylab = "Mean log(proportion explored)", trace.label = "Algorithm")

In [ ]:
interaction.plot(dat$S, dat$D, dat$y, xlab = "Size", ylab = "Mean log(proportion explored)", trace.label = "Density")

In [ ]:
boxplot(frac_explored ~ A, data = dat, ylab = "Proportion of cells explored")

## 3. Algorithm comparisons

Algorithm comparisons use only within-maze differences, so they are tested against the subplot error. Fitting with maze as a block gives the same error term.

In [ ]:
fit_w <- lm(y ~ maze + A*(S + D), data = dat)
emm <- emmeans(fit_w, ~ A)

### Planned contrasts

Level order: bfs, dijkstra, astar, greedy, random_walk. BFS and Dijkstra are expected to be nearly identical on this unit-cost grid.

In [ ]:
contrast(emm, method = list(
  "A* vs Dijkstra"          = c(0, -1, 1, 0, 0),
  "informed vs uninformed"  = c(-1/2, -1/2, 1/2, 1/2, 0),
  "random walk vs the rest" = c(-1/4, -1/4, -1/4, -1/4, 1)
))

### Tukey multiple comparisons

Overall, and within each size and density.

In [ ]:
pairs(emm, adjust = "tukey")

In [ ]:
pairs(emmeans(fit_w, ~ A | S), adjust = "tukey")

In [ ]:
pairs(emmeans(fit_w, ~ A | D), adjust = "tukey")

## 4. Diagnostics

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4)
par(mfrow = c(1, 3))
qqnorm(resid(fit[["maze"]]),   main = "Whole-plot residuals"); qqline(resid(fit[["maze"]]))
qqnorm(resid(fit[["Within"]]), main = "Subplot residuals");    qqline(resid(fit[["Within"]]))
plot(fitted(fit_w), resid(fit_w), xlab = "Fitted", ylab = "Residual",
     main = "Residuals vs fitted")
par(mfrow = c(1, 1))
options(repr.plot.width = 9, repr.plot.height = 5)

In [ ]:
boxplot(resid(fit_w) ~ dat$A, ylab = "Residual", main = "Residuals by algorithm")

### Is the log transformation enough?

Look for a Box-Cox lambda near 0.

In [ ]:
boxcox(lm(frac_explored ~ maze + A*(S + D), data = dat))

## 5. Sensitivity check

Refit without random walk, since its runs can be capped.

In [ ]:
dat5 <- droplevels(dat[dat$A != "random_walk", ])
summary(aov(y ~ G + S*D + A*(S + D) + Error(maze), data = dat5))